# Milestone 3 on Kaggle: train the models to deploy, and keep their weights

Settings: **Accelerator = GPU T4 x2** (only the first GPU is used), **Internet = on**. Run with *Save Version > Save & Run All*.

The milestone 2 runs kept metrics only. This notebook produces the checkpoints that will be exported to ONNX and measured on the Raspberry Pi 5:

1. clones the repository, checks the frozen split, downloads EuroSAT RGB (MIT license, MD5 checked);
2. trains the two new deployment candidates at 100 % of the data with 3 seeds, like every other config: ResNet-18 fine-tuned, and ResNet-50 fine-tuned at 112 px. The seed 0 weights are saved;
3. retrains seed 0 at 100 % of the three configs already measured (scratch CNN, ResNet-50 and ViT-S fine-tuned), this time saving the weights. These runs go to `deploy_training/`, apart from the milestone 2 results, because GPU training is not bit-for-bit reproducible and their accuracy may differ slightly from the published seed 0;
4. zips `results.zip` (JSON, small) and `checkpoints.zip` (about 330 MB). Download both from the Output tab.

In [ ]:
import os, shutil, subprocess, torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')

def run(cmd):
    """Run a shell command, stream its output, raise if it fails."""
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
run('rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/eurosat-onboard-transfer.git /tmp/repo')
os.chdir('/tmp/repo')
os.environ['PYTHONPATH'] = 'src'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
run('git log -1 --format="commit %h %s"')
run('python -m pytest -q tests/test_splits.py')
run('python scripts/download_eurosat.py --data-dir /tmp/data')
DATA = '/tmp/data/EuroSAT_RGB/EuroSAT_RGB'
W = '/kaggle/working'

In [ ]:
def zip_outputs():
    run(f'cd {W} && rm -f results.zip checkpoints.zip && zip -qr results.zip results'
        f' && (test -d checkpoints && zip -qr checkpoints.zip checkpoints || true) && ls -lh {W}/*.zip')

for cfg in ['resnet18_ft', 'resnet50_ft_112']:
    run(f'python scripts/run_experiment.py --config configs/{cfg}.yaml --pcts 100 --seeds 0 1 2 '
        f'--data-root {DATA} --out-dir {W}/results --checkpoint-dir {W}/checkpoints --checkpoint-seeds 0')
    zip_outputs()

In [ ]:
for cfg in ['cnn_scratch', 'resnet50_ft', 'vit_small_ft']:
    run(f'python scripts/run_experiment.py --config configs/{cfg}.yaml --pcts 100 --seeds 0 '
        f'--data-root {DATA} --out-dir {W}/results/deploy_training --checkpoint-dir {W}/checkpoints --checkpoint-seeds 0')
    zip_outputs()
run(f'ls -lh {W}/checkpoints')